# ❌⭕ Project 02 — Unbeatable Tic-Tac-Toe (Recursion + Minimax AI)

**Skills used:** lists, tuples, recursion, backtracking, `functools.cache`, classes, simulation & testing.

**Big idea — Minimax:** assume both players play perfectly. For the AI's turn pick the move with the *highest* score; for the opponent's turn assume they pick the *lowest*. Terminal scores: `+1` AI wins, `0` draw, `−1` AI loses. Recursion explores every possible future.

---
## Step 1 — Board representation and rules
The board is a `tuple` of 9 cells (immutable → hashable → cacheable).

In [1]:
from functools import cache

LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),      # rows
         (0, 3, 6), (1, 4, 7), (2, 5, 8),      # columns
         (0, 4, 8), (2, 4, 6)]                 # diagonals
EMPTY = " "

def new_board():
    return (EMPTY,) * 9

def winner(board):
    for a, b, c in LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    return None

def moves(board):
    return [i for i, cell in enumerate(board) if cell == EMPTY]

def play(board, index, player):
    return board[:index] + (player,) + board[index + 1:]       # returns a NEW board

def other(player):
    return "O" if player == "X" else "X"

def render(board):
    rows = [" | ".join(board[i:i + 3]) for i in (0, 3, 6)]
    return "\n---------\n".join(rows)

b = play(play(new_board(), 4, "X"), 0, "O")
print(render(b), "\nfree:", moves(b), "| winner:", winner(b))

O |   |  
---------
  | X |  
---------
  |   |   
free: [1, 2, 3, 5, 6, 7, 8] | winner: None


## Step 2 — Minimax

In [2]:
@cache                                   # memoisation: each position is solved once, however many paths reach it
def minimax(board, player, ai):
    """Score of `board` for `ai` when `player` is about to move (+1 win, 0 draw, -1 loss)."""
    w = winner(board)
    if w == ai:
        return 1
    if w is not None:
        return -1
    free = moves(board)
    if not free:
        return 0
    scores = (minimax(play(board, m, player), other(player), ai) for m in free)
    return max(scores) if player == ai else min(scores)

def best_move(board, ai):
    """Choose the move with the best minimax score (ties: first found)."""
    free = moves(board)
    return max(free, key=lambda m: minimax(play(board, m, ai), other(ai), ai))

# The AI as "O" answers X's corner opening with the centre (the only non-losing reply)
opening = play(new_board(), 0, "X")
print("O replies at:", best_move(opening, "O"))
print("positions evaluated & cached:", minimax.cache_info().currsize)

O replies at: 4
positions evaluated & cached: 1869


## Step 3 — Prove it's unbeatable
Let the AI (as O) face **every possible sequence of human moves** as X, and count outcomes. If the AI is truly perfect, X should *never* win.

In [3]:
def explore(board, human, ai, to_move, results):
    w = winner(board)
    if w or not moves(board):
        results[w or "draw"] = results.get(w or "draw", 0) + 1
        return
    if to_move == ai:
        explore(play(board, best_move(board, ai), ai), human, ai, human, results)
    else:
        for m in moves(board):                         # human tries EVERY move
            explore(play(board, m, human), human, ai, ai, results)

as_second = {}
explore(new_board(), "X", "O", "X", as_second)
print("AI plays second (O): ", as_second)          # X never wins

as_first = {}
explore(new_board(), "O", "X", "X", as_first)       # AI is X and moves first
print("AI plays first  (X): ", as_first)            # O never wins
assert "X" not in as_second and "O" not in as_first

AI plays second (O):  {'O': 498, 'draw': 183}
AI plays first  (X):  {'X': 99, 'draw': 2}


## Step 4 — The game loop (interactive)
`play_game()` reads the human's cell numbers 1–9. Here we also give a scripted mode to test without typing.

In [4]:
def play_game(human="X", moves_script=None):
    ai = other(human)
    board, turn = new_board(), "X"
    script = iter(moves_script) if moves_script is not None else None
    while winner(board) is None and moves(board):
        if turn == human:
            if script is not None:                      # scripted human: next legal scripted move, else first free cell
                cell = next((c - 1 for c in script if c - 1 in moves(board)), moves(board)[0])
            else:
                cell = int(input("Your move (1-9): ")) - 1
                if cell not in moves(board):
                    print("Cell taken or invalid, try again."); continue
            board = play(board, cell, human)
        else:
            board = play(board, best_move(board, ai), ai)
        turn = other(turn)
    print(render(board))
    w = winner(board)
    print("Winner:", w if w else "draw")
    return w

play_game("X", moves_script=[1, 3, 8, 6, 5])       # a scripted human: the AI never loses

X | O | X
---------
O | O | X
---------
X | X | O
Winner: draw


In [ ]:
# notest
# play_game("X")     # <- run this line in a terminal/notebook for a real game against the AI

## Step 5 — Speed-up: alpha-beta pruning (optional deep-dive)
Minimax explores every branch; **alpha-beta** skips branches that can't change the decision. For Tic-Tac-Toe the cache is enough, but for Connect-4 or chess you need pruning.

In [5]:
def alphabeta(board, player, ai, alpha=-2, beta=2):
    w = winner(board)
    if w == ai: return 1
    if w is not None: return -1
    free = moves(board)
    if not free: return 0
    if player == ai:
        best = -2
        for m in free:
            best = max(best, alphabeta(play(board, m, player), other(player), ai, alpha, beta))
            alpha = max(alpha, best)
            if alpha >= beta: break          # prune
        return best
    best = 2
    for m in free:
        best = min(best, alphabeta(play(board, m, player), other(player), ai, alpha, beta))
        beta = min(beta, best)
        if alpha >= beta: break
    return best

import time
def count_nodes():
    calls = 0
    def counter(board, player, ai):
        nonlocal calls
        calls += 1
        w = winner(board)
        if w == ai: return 1
        if w is not None: return -1
        free = moves(board)
        if not free: return 0
        vals = [counter(play(board, m, player), other(player), ai) for m in free]
        return max(vals) if player == ai else min(vals)
    counter(new_board(), "X", "X")
    return calls

t = time.perf_counter(); plain = count_nodes(); t1 = time.perf_counter() - t
t = time.perf_counter(); v = alphabeta(new_board(), "X", "X"); t2 = time.perf_counter() - t
print(f"plain minimax visits {plain:,} positions in {t1:.2f}s; alpha-beta value={v} in {t2:.2f}s")
print("Perfect play from the empty board is a draw:", v == 0)

plain minimax visits 549,946 positions in 1.30s; alpha-beta value=0 in 0.06s
Perfect play from the empty board is a draw: True


---
## 🚀 Extensions
1. Make the AI *imperfect* on purpose (pick a random move 20 % of the time) — "easy/medium/hard" levels.
2. Generalise to **4×4** boards. What breaks? (Hint: state space explodes — add depth limits + heuristics.)
3. Build a GUI with `tkinter` (standard library) reusing all the functions above.
4. Implement **Connect Four** with alpha-beta and a depth limit.

**What you practised:** recursion, memoisation, game trees, immutability for safe caching, exhaustive testing.